# Catchment carbon benefits for mangroves and forest restoration

This notebook aggregates global carbon-benefit estimates for mangrove patches and forest restoration areas to the same Paper 2 river-catchment geography used for river-flood avoided-EAD and prioritisation analysis.

The outputs are intended as prioritisation inputs rather than a fixed prioritisation scheme: the notebook keeps carbon totals, carbon densities, Paper 2 river-flood avoided EAD/BCR metrics, and mangrove coastal avoided-EAD metrics as separate fields.

In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

base_path = Path.home() / "Documents/Geospatial_analysis/dphil_papers"
library_path = base_path / "robyns_libraries"
if str(library_path) not in sys.path:
    sys.path.append(str(library_path))

import Robyn_paper_2_defs

Robyn_paper_2_defs.set_nature_style()
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
catchments_path = base_path / "dphil_paper_2/processed_data/major_river_catchments/major_basins_plus_coastal_unionized_final.gpkg"
mangrove_carbon_path = base_path / "dphil_paper_3/processed_data/carbon/global_carbon_benefit/fn_mangrove_patch_global_carbon_benefit_summary.gpkg"
forest_catchment_carbon_path = base_path / "dphil_paper_3/processed_data/carbon/global_carbon_benefit_restorable_forests/restorable_forest_global_carbon_benefit_by_catchment.gpkg"
forest_catchment_accounting_path = base_path / "dphil_paper_3/processed_data/carbon/global_carbon_benefit_restorable_forests/restorable_forest_global_carbon_benefit_by_catchment.csv"

output_data_dir = base_path / "dphil_paper_3/processed_data/carbon/catchment_carbon_benefits"
figure_dir = base_path / "dphil_paper_3/results/co_benefits/carbon/catchment_carbon_benefits"
output_data_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

combined_catchment_csv_path = output_data_dir / "catchment_carbon_benefits_mangroves_forests.csv"
combined_catchment_gpkg_path = output_data_dir / "catchment_carbon_benefits_mangroves_forests.gpkg"
mangrove_catchment_csv_path = output_data_dir / "mangrove_carbon_by_paper2_catchment.csv"
mangrove_catchment_gpkg_path = output_data_dir / "mangrove_carbon_by_paper2_catchment.gpkg"
mangrove_allocation_csv_path = output_data_dir / "mangrove_patch_to_paper2_catchment_allocation.csv"
summary_csv_path = output_data_dir / "catchment_carbon_benefits_key_summary.csv"
map_png_path = figure_dir / "catchment_carbon_benefits_mangroves_forests_panel_map.png"
map_pdf_path = figure_dir / "catchment_carbon_benefits_mangroves_forests_panel_map.pdf"

for required_path in [catchments_path, mangrove_carbon_path, forest_catchment_carbon_path, forest_catchment_accounting_path]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

print(f"Catchments: {catchments_path}")
print(f"Mangrove carbon: {mangrove_carbon_path}")
print(f"Forest catchment carbon: {forest_catchment_carbon_path}")
print(f"Forest catchment accounting CSV: {forest_catchment_accounting_path}")
print(f"Outputs: {output_data_dir}")

## Load inputs

The catchment layer is the Paper 2 unionised major-river-plus-coastal catchment geography. Forest restoration carbon is already available by this catchment geography from the restorable-forest carbon notebook. Mangrove carbon is allocated below using patch/catchment spatial intersections.

In [ ]:
catchments = gpd.read_file(catchments_path)
mangroves = gpd.read_file(mangrove_carbon_path).to_crs(catchments.crs)
forest_catchments = gpd.read_file(forest_catchment_carbon_path).to_crs(catchments.crs)
forest_catchment_accounting = pd.read_csv(forest_catchment_accounting_path)

catchments["catchment_uid"] = catchments["catchment_uid"].astype(int)
forest_catchments["catchment_uid"] = forest_catchments["catchment_uid"].astype(int)

print(f"Catchments loaded: {len(catchments):,}")
print(f"Mangrove patches loaded: {len(mangroves):,}")
print(f"Forest catchment rows loaded: {len(forest_catchments):,}")
print(f"Forest accounting rows loaded: {len(forest_catchment_accounting):,}")
print(f"CRS: {catchments.crs}")

In [ ]:
def fill_numeric_columns(dataframe, column_names):
    cleaned = dataframe.copy()
    for column_name in column_names:
        cleaned[column_name] = pd.to_numeric(cleaned[column_name], errors="coerce").fillna(0)
    return cleaned


def rank_highest_first(values):
    return values.rank(method="min", ascending=False).astype("Int64")


def add_share_column(dataframe, value_column, share_column):
    total_value = dataframe[value_column].sum()
    dataframe[share_column] = np.where(total_value > 0, dataframe[value_column] / total_value * 100, 0)


def divide_or_zero(numerator, denominator):
    return np.where(denominator > 0, numerator / denominator, 0)


def style_jamaica_axis(axis, title_text, total_bounds):
    min_x, min_y, max_x, max_y = total_bounds
    x_padding = (max_x - min_x) * 0.04
    y_padding = (max_y - min_y) * 0.08
    axis.set_xlim(min_x - x_padding, max_x + x_padding)
    axis.set_ylim(min_y - y_padding, max_y + y_padding)
    axis.set_title(title_text, pad=3)
    axis.set_axis_off()
    Robyn_paper_2_defs.draw_scale_bar(
        axis,
        location=(0.88, 0.78),
        length_km=20,
        linewidth=0.6,
        label_offset=0.02,
        km_offset=0.01,
    )
    Robyn_paper_2_defs.draw_north_arrow(axis, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)

## Allocate mangrove patch carbon to Paper 2 catchments

Mangrove patches are coastal and can cross catchment boundaries or extend beyond the land/catchment polygon. To avoid losing patch carbon, the notebook uses the spatial-intersection area to calculate each patch's catchment weights and then normalises weights to sum to 1 for each patch. Patches with no polygon-area intersection are assigned to their nearest Paper 2 catchment.

In [ ]:
mangrove_numeric_columns = [
    "patch_area_ha",
    "carbon_benefit_per_patch_ha_with_nn_fill",
    "total_carbon_benefit_with_nn_fill",
    "avoided_ead_usd_min",
    "avoided_ead_usd_max",
    "patch_positive_avoided_EADs",
]
mangroves = fill_numeric_columns(mangroves, mangrove_numeric_columns)
mangroves["mangrove_patch_area_ha"] = np.where(
    mangroves["patch_area_ha"] > 0,
    mangroves["patch_area_ha"],
    mangroves.geometry.area / 10_000,
)

mangrove_source_columns = [
    "Mangrove_ID",
    "mangrove_patch_area_ha",
    "carbon_benefit_per_patch_ha_with_nn_fill",
    "total_carbon_benefit_with_nn_fill",
    "avoided_ead_usd_min",
    "avoided_ead_usd_max",
    "patch_positive_avoided_EADs",
    "geometry",
]

mangrove_intersections = gpd.overlay(
    mangroves[mangrove_source_columns],
    catchments[["catchment_uid", "geometry"]],
    how="intersection",
    keep_geom_type=True,
)
mangrove_intersections["physical_intersection_area_ha"] = mangrove_intersections.geometry.area / 10_000
mangrove_intersections = mangrove_intersections[mangrove_intersections["physical_intersection_area_ha"] > 0].copy()

mangrove_overlap_totals = (
    mangrove_intersections.groupby("Mangrove_ID", as_index=False)["physical_intersection_area_ha"]
    .sum()
    .rename(columns={"physical_intersection_area_ha": "patch_intersection_total_ha"})
)
mangrove_allocations = mangrove_intersections.merge(mangrove_overlap_totals, on="Mangrove_ID", how="left")
mangrove_allocations["allocation_weight"] = (
    mangrove_allocations["physical_intersection_area_ha"] / mangrove_allocations["patch_intersection_total_ha"]
)
mangrove_allocations["allocation_method"] = "area_normalised_intersection"

intersecting_mangrove_ids = set(mangrove_allocations["Mangrove_ID"])
missing_mangroves = mangroves[~mangroves["Mangrove_ID"].isin(intersecting_mangrove_ids)].copy()

if not missing_mangroves.empty:
    nearest_catchments = gpd.sjoin_nearest(
        missing_mangroves[mangrove_source_columns],
        catchments[["catchment_uid", "geometry"]],
        how="left",
        distance_col="nearest_catchment_distance_m",
    )
    nearest_catchments["physical_intersection_area_ha"] = 0.0
    nearest_catchments["patch_intersection_total_ha"] = 0.0
    nearest_catchments["allocation_weight"] = 1.0
    nearest_catchments["allocation_method"] = "nearest_catchment_no_intersection"
    nearest_catchments = nearest_catchments.drop(columns=["index_right"])
    mangrove_allocation_rows = pd.concat(
        [
            pd.DataFrame(mangrove_allocations.drop(columns="geometry")),
            pd.DataFrame(nearest_catchments.drop(columns="geometry")),
        ],
        ignore_index=True,
        sort=False,
    )
else:
    mangrove_allocation_rows = pd.DataFrame(mangrove_allocations.drop(columns="geometry"))

mangrove_allocation_rows["allocated_mangrove_area_ha"] = (
    mangrove_allocation_rows["mangrove_patch_area_ha"] * mangrove_allocation_rows["allocation_weight"]
)
mangrove_allocation_rows["allocated_mangrove_carbon_tonnes_c"] = (
    mangrove_allocation_rows["total_carbon_benefit_with_nn_fill"] * mangrove_allocation_rows["allocation_weight"]
)
mangrove_allocation_rows["allocated_mangrove_coastal_avoided_ead_usd_min"] = (
    mangrove_allocation_rows["avoided_ead_usd_min"] * mangrove_allocation_rows["allocation_weight"]
)
mangrove_allocation_rows["allocated_mangrove_coastal_avoided_ead_usd_max"] = (
    mangrove_allocation_rows["avoided_ead_usd_max"] * mangrove_allocation_rows["allocation_weight"]
)
mangrove_allocation_rows["allocated_mangrove_coastal_avoided_ead_usd_mean"] = (
    mangrove_allocation_rows["patch_positive_avoided_EADs"] * mangrove_allocation_rows["allocation_weight"]
)

mangrove_by_catchment = mangrove_allocation_rows.groupby("catchment_uid", as_index=False).agg(
    mangrove_unique_patch_count=("Mangrove_ID", "nunique"),
    mangrove_patch_equivalent_count=("allocation_weight", "sum"),
    mangrove_physical_intersection_area_ha=("physical_intersection_area_ha", "sum"),
    mangrove_area_ha=("allocated_mangrove_area_ha", "sum"),
    mangrove_carbon_tonnes_c=("allocated_mangrove_carbon_tonnes_c", "sum"),
    mangrove_coastal_avoided_ead_usd_min=("allocated_mangrove_coastal_avoided_ead_usd_min", "sum"),
    mangrove_coastal_avoided_ead_usd_mean=("allocated_mangrove_coastal_avoided_ead_usd_mean", "sum"),
    mangrove_coastal_avoided_ead_usd_max=("allocated_mangrove_coastal_avoided_ead_usd_max", "sum"),
)
mangrove_by_catchment["mangrove_carbon_tonnes_c_per_ha"] = divide_or_zero(
    mangrove_by_catchment["mangrove_carbon_tonnes_c"],
    mangrove_by_catchment["mangrove_area_ha"],
)

allocation_checks = pd.DataFrame(
    [
        {
            "metric": "mangrove_area_ha",
            "source_total": mangroves["mangrove_patch_area_ha"].sum(),
            "allocated_total": mangrove_by_catchment["mangrove_area_ha"].sum(),
        },
        {
            "metric": "mangrove_carbon_tonnes_c",
            "source_total": mangroves["total_carbon_benefit_with_nn_fill"].sum(),
            "allocated_total": mangrove_by_catchment["mangrove_carbon_tonnes_c"].sum(),
        },
        {
            "metric": "mangrove_coastal_avoided_ead_usd_mean",
            "source_total": mangroves["patch_positive_avoided_EADs"].sum(),
            "allocated_total": mangrove_by_catchment["mangrove_coastal_avoided_ead_usd_mean"].sum(),
        },
    ]
)
allocation_checks["difference"] = allocation_checks["allocated_total"] - allocation_checks["source_total"]

print(f"Mangrove allocation rows: {len(mangrove_allocation_rows):,}")
print(f"Mangrove patches assigned by nearest catchment: {len(missing_mangroves):,}")
display(allocation_checks)

## Combine forest restoration and mangrove carbon by catchment

Forest carbon comes from the existing catchment-level restorable-forest carbon output and retains the Paper 2 river-flood avoided-EAD and BCR fields. Mangrove carbon is added as a separate coastal-flood co-benefit field. The notebook does not add coastal and river avoided EADs together; they are kept as separate prioritisation inputs.

In [ ]:
forest_metric_columns = {
    "patch_count": "forest_patch_count",
    "observed_patch_count": "forest_observed_patch_count",
    "restorable_area_ha": "forest_restorable_area_ha",
    "total_carbon_potential_source_units": "forest_carbon_tonnes_c_observed_only",
    "total_carbon_potential_with_nn_fill": "forest_carbon_tonnes_c",
    "filled_patch_count": "forest_patches_with_nearest_neighbour_fill",
    "carbon_potential_per_restorable_ha_with_nn_fill": "forest_carbon_tonnes_c_per_ha",
    "paper2_priority_area_ha_min": "paper2_priority_area_ha_min",
    "paper2_priority_area_ha_max": "paper2_priority_area_ha_max",
    "paper2_costs_usd_min": "paper2_costs_usd_min",
    "paper2_costs_usd_max": "paper2_costs_usd_max",
    "paper2_avoided_ead_usd_min": "paper2_river_avoided_ead_usd_min",
    "paper2_avoided_ead_usd_max": "paper2_river_avoided_ead_usd_max",
    "paper2_avoided_ead_usd_discounted_min": "paper2_river_avoided_ead_usd_discounted_min",
    "paper2_avoided_ead_usd_discounted_max": "paper2_river_avoided_ead_usd_discounted_max",
    "paper2_total_cost_usd_discounted_min": "paper2_total_cost_usd_discounted_min",
    "paper2_total_cost_usd_discounted_max": "paper2_total_cost_usd_discounted_max",
    "paper2_bcr_usd_discounted_min": "paper2_bcr_usd_discounted_min",
    "paper2_bcr_usd_discounted_max": "paper2_bcr_usd_discounted_max",
    "carbon_potential_per_usd_cost_min": "forest_carbon_tonnes_c_per_usd_cost_min",
    "carbon_potential_per_usd_cost_max": "forest_carbon_tonnes_c_per_usd_cost_max",
}

forest_metrics = forest_catchments[["catchment_uid", *forest_metric_columns.keys()]].rename(columns=forest_metric_columns)
catchment_carbon = catchments.merge(forest_metrics, on="catchment_uid", how="left")
catchment_carbon = catchment_carbon.merge(mangrove_by_catchment, on="catchment_uid", how="left")

combined_numeric_columns = [
    column_name for column_name in catchment_carbon.columns if column_name != "geometry" and column_name != "catchment_uid"
]
catchment_carbon = fill_numeric_columns(catchment_carbon, combined_numeric_columns)

catchment_carbon["combined_nbs_area_ha"] = (
    catchment_carbon["forest_restorable_area_ha"] + catchment_carbon["mangrove_area_ha"]
)
catchment_carbon["combined_nbs_carbon_tonnes_c"] = (
    catchment_carbon["forest_carbon_tonnes_c"] + catchment_carbon["mangrove_carbon_tonnes_c"]
)
catchment_carbon["combined_nbs_carbon_tonnes_c_per_ha"] = divide_or_zero(
    catchment_carbon["combined_nbs_carbon_tonnes_c"],
    catchment_carbon["combined_nbs_area_ha"],
)
catchment_carbon["mangrove_share_of_combined_nbs_carbon_pct"] = divide_or_zero(
    catchment_carbon["mangrove_carbon_tonnes_c"],
    catchment_carbon["combined_nbs_carbon_tonnes_c"],
) * 100
catchment_carbon["forest_share_of_combined_nbs_carbon_pct"] = divide_or_zero(
    catchment_carbon["forest_carbon_tonnes_c"],
    catchment_carbon["combined_nbs_carbon_tonnes_c"],
) * 100

for value_column in [
    "forest_carbon_tonnes_c",
    "mangrove_carbon_tonnes_c",
    "combined_nbs_carbon_tonnes_c",
    "paper2_river_avoided_ead_usd_min",
    "paper2_river_avoided_ead_usd_max",
    "mangrove_coastal_avoided_ead_usd_mean",
]:
    add_share_column(catchment_carbon, value_column, f"{value_column}_national_share_pct")
    catchment_carbon[f"{value_column}_rank_desc"] = rank_highest_first(catchment_carbon[value_column])

key_columns = [
    "catchment_uid",
    "area_ha",
    "forest_restorable_area_ha",
    "forest_carbon_tonnes_c",
    "forest_carbon_tonnes_c_per_ha",
    "mangrove_area_ha",
    "mangrove_carbon_tonnes_c",
    "mangrove_carbon_tonnes_c_per_ha",
    "combined_nbs_area_ha",
    "combined_nbs_carbon_tonnes_c",
    "combined_nbs_carbon_tonnes_c_per_ha",
    "paper2_river_avoided_ead_usd_min",
    "paper2_river_avoided_ead_usd_max",
    "paper2_bcr_usd_discounted_min",
    "paper2_bcr_usd_discounted_max",
    "mangrove_coastal_avoided_ead_usd_mean",
]

display(catchment_carbon[key_columns].sort_values("combined_nbs_carbon_tonnes_c", ascending=False).head(12))

In [ ]:
forest_accounting_total_carbon = pd.to_numeric(
    forest_catchment_accounting["total_carbon_potential_with_nn_fill"], errors="coerce"
).fillna(0).sum()
forest_unassigned_or_unmapped_carbon = forest_accounting_total_carbon - catchment_carbon["forest_carbon_tonnes_c"].sum()

summary_rows = [
    {"metric": "paper2_catchments", "value": len(catchment_carbon)},
    {"metric": "forest_restorable_area_ha", "value": catchment_carbon["forest_restorable_area_ha"].sum()},
    {"metric": "forest_carbon_tonnes_c_mapped_to_catchments", "value": catchment_carbon["forest_carbon_tonnes_c"].sum()},
    {"metric": "forest_carbon_tonnes_c_accounting_total", "value": forest_accounting_total_carbon},
    {"metric": "forest_carbon_tonnes_c_unassigned_or_unmapped", "value": forest_unassigned_or_unmapped_carbon},
    {"metric": "mangrove_area_ha", "value": catchment_carbon["mangrove_area_ha"].sum()},
    {"metric": "mangrove_carbon_tonnes_c", "value": catchment_carbon["mangrove_carbon_tonnes_c"].sum()},
    {"metric": "combined_nbs_area_ha", "value": catchment_carbon["combined_nbs_area_ha"].sum()},
    {"metric": "combined_nbs_carbon_tonnes_c", "value": catchment_carbon["combined_nbs_carbon_tonnes_c"].sum()},
    {"metric": "paper2_river_avoided_ead_usd_min", "value": catchment_carbon["paper2_river_avoided_ead_usd_min"].sum()},
    {"metric": "paper2_river_avoided_ead_usd_max", "value": catchment_carbon["paper2_river_avoided_ead_usd_max"].sum()},
    {"metric": "mangrove_coastal_avoided_ead_usd_mean", "value": catchment_carbon["mangrove_coastal_avoided_ead_usd_mean"].sum()},
]
summary_table = pd.DataFrame(summary_rows)
summary_table.to_csv(summary_csv_path, index=False)

print("Key national totals after catchment aggregation")
display(summary_table)

print("Top catchments by combined NbS carbon")
display(
    catchment_carbon[key_columns]
    .sort_values("combined_nbs_carbon_tonnes_c", ascending=False)
    .head(10)
)

print("Top catchments by mangrove carbon")
display(
    catchment_carbon[key_columns]
    .sort_values("mangrove_carbon_tonnes_c", ascending=False)
    .head(10)
)

## Save prioritisation input tables

The GeoPackage and CSV contain the same catchment-level metrics. The GeoPackage retains geometry for mapping; the CSV is easier to inspect or use in later weighting/scoring notebooks.

In [ ]:
catchment_carbon.to_file(combined_catchment_gpkg_path, driver="GPKG")
catchment_carbon.drop(columns="geometry").to_csv(combined_catchment_csv_path, index=False)

mangrove_catchment_gdf = catchments.merge(mangrove_by_catchment, on="catchment_uid", how="left")
mangrove_numeric_output_columns = [
    column_name for column_name in mangrove_catchment_gdf.columns if column_name != "geometry" and column_name != "catchment_uid"
]
mangrove_catchment_gdf = fill_numeric_columns(mangrove_catchment_gdf, mangrove_numeric_output_columns)
mangrove_catchment_gdf.to_file(mangrove_catchment_gpkg_path, driver="GPKG")
mangrove_catchment_gdf.drop(columns="geometry").to_csv(mangrove_catchment_csv_path, index=False)

mangrove_allocation_rows.to_csv(mangrove_allocation_csv_path, index=False)

print(f"Saved combined catchment CSV: {combined_catchment_csv_path}")
print(f"Saved combined catchment GeoPackage: {combined_catchment_gpkg_path}")
print(f"Saved mangrove catchment CSV: {mangrove_catchment_csv_path}")
print(f"Saved mangrove catchment GeoPackage: {mangrove_catchment_gpkg_path}")
print(f"Saved mangrove allocation table: {mangrove_allocation_csv_path}")
print(f"Saved summary table: {summary_csv_path}")

## Catchment maps

These maps show the same catchment geography with forest restoration carbon potential, mangrove carbon benefit, and combined NbS carbon. They are diagnostic maps for exploring future prioritisation weights.

In [ ]:
map_columns = [
    ("forest_carbon_tonnes_c", "a) Forest restoration carbon potential", "Forest restoration carbon (tonnes C)", "YlGn"),
    ("mangrove_carbon_tonnes_c", "b) Mangrove carbon benefit", "Mangrove carbon (tonnes C)", "PuBuGn"),
    ("combined_nbs_carbon_tonnes_c", "c) Combined NbS carbon", "Combined carbon (tonnes C)", "viridis"),
]

with mpl.rc_context(Robyn_paper_2_defs.NATURE_RC):
    figure, axes = plt.subplots(3, 1, figsize=(180 / 25.4, 230 / 25.4), constrained_layout=True)
    total_bounds = catchments.total_bounds

    for axis, (column_name, title_text, legend_label, colour_map) in zip(axes, map_columns):
        catchment_carbon.plot(
            column=column_name,
            ax=axis,
            cmap=colour_map,
            linewidth=0.25,
            edgecolor="#4D4D4D",
            legend=True,
            legend_kwds={"label": legend_label, "shrink": 0.55, "pad": 0.01},
            missing_kwds={"color": "#F2F2F2", "edgecolor": "#BDBDBD"},
        )
        style_jamaica_axis(axis, title_text, total_bounds)

    figure.savefig(map_png_path, dpi=300, bbox_inches="tight")
    figure.savefig(map_pdf_path, bbox_inches="tight")
    plt.show()

print(f"Saved map PNG: {map_png_path}")
print(f"Saved map PDF: {map_pdf_path}")

In [ ]:
output_index = pd.DataFrame(
    [
        {"description": "combined catchment prioritisation input table", "path": combined_catchment_csv_path},
        {"description": "combined catchment prioritisation input map layer", "path": combined_catchment_gpkg_path},
        {"description": "mangrove-only catchment summary", "path": mangrove_catchment_csv_path},
        {"description": "mangrove patch-to-catchment allocation table", "path": mangrove_allocation_csv_path},
        {"description": "key national summary table", "path": summary_csv_path},
        {"description": "catchment carbon panel map PNG", "path": map_png_path},
        {"description": "catchment carbon panel map PDF", "path": map_pdf_path},
    ]
)
display(output_index)